### Dataset Analysis

Displaying the root structure of the dataset.


In [ ]:
from pathlib import Path

space =  '    '; branch = '│   '; tee =    '├── '; last =   '└── '
def tree(dir_path: Path, prefix: str='', exclude_extensions=None):
    if exclude_extensions is None: exclude_extensions = ['.jpg']
    if isinstance(dir_path, str): dir_path = Path(dir_path)
    # Filter out files with excluded extensions
    contents = [p for p in dir_path.iterdir()  if not (p.is_file() and p.suffix.lower() in exclude_extensions)]
    if not contents: return
    # contents each get pointers that are ├── with a final └── :
    pointers = [tee] * (len(contents) - 1) + [last]
    for pointer, path in zip(pointers, contents):
        yield prefix + pointer + path.name
        if path.is_dir():  # extend the prefix and recurse:
            extension = branch if pointer == tee else space
            # i.e. space because last, └── , above so no more |
            yield from tree(path, prefix=prefix+extension, exclude_extensions=exclude_extensions)

for line in tree('./nikitarom'):
    print(line)

└── planets-dataset
    ├── 3.complete
    └── versions
        └── 3
            ├── planet
            │   └── planet
            │       ├── train_classes.csv
            │       ├── test-jpg
            │       ├── sample_submission.csv
            │       └── train-jpg
            └── test-jpg-additional
                └── test-jpg-additional


The first CSV table named **sample_submission.csv** contains 61,191 values and has 2 columns: image_name - the name of the given image; tags - a description of the properties of what is in the images.

The second CSV table named **train_classes.csv** contains 40,479 values and also has 2 columns: image_name - the name of the given image; tags - a description of the properties of what is in the images, e.g.: clear_primary, clear_cloudy_primary, etc...

We have directories for images:

**test-jpg**, which contains test images, approximately 40,000 of them.

**train-jpg**, which contains training images, approximately 40,000 of them.

**test-jpg-additional**, which contains approximately 20,500 additional test images.


### Loading Data for the Training Model


In [ ]:
# import of required datasets
import os
import pandas as pd
import numpy as np
import seaborn as sns
import time
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, multilabel_confusion_matrix, classification_report, confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Setting paths to data
DATA_DIR = './nikitarom/planets-dataset/versions/3/'
TRAIN_DIR = os.path.join(DATA_DIR, 'planet/planet/train-jpg')
TEST_DIR = os.path.join(DATA_DIR, 'planet/planet/test-jpg')
TRAIN_CLASSES = os.path.join(DATA_DIR, 'planet/planet/train_classes.csv')
SUBMISSION = os.path.join(DATA_DIR, 'planet/planet/sample_submission.csv')

# Loading CSV files
train_df = pd.read_csv(TRAIN_CLASSES)
submission_df = pd.read_csv(SUBMISSION)


ModuleNotFoundError: No module named 'pandas'

I imported the required datasets for working with data. I defined the paths to the directories containing training and test images and image descriptions. I loaded the CSV files.

#### Retrieving Table Information


In [ ]:
print(f"Number of training records: {len(train_df)}")
print(f"Number of test records: {len(submission_df)}")

# Display information about the table
print('\nDisplaying information about the training table:')
print("Table header: \n", train_df.head(), '\n')
print("Table info: ", train_df.info(), '\n')
print("Null values: ", train_df.isnull().sum(), '\n')

# Display the tag distribution in the 'tags' column
all_tags = []
for tags in train_df['tags'].values:
    all_tags.extend(tags.split())
unique_tags = sorted(list(set(all_tags)))
print(f"\nNumber of unique tags: {len(unique_tags)}")
print(f"Unique tags: {unique_tags}")


I printed out the table information to know how many records it has, how the columns are configured, whether null values are allowed, and what types they are. I also printed out information about whether there are any null values and how many tags there are in the *tags* column, where I found that 17 different tags are used to describe the images.

Unique tags:

agriculture, artisinal_mine, bare_ground, blooming, blow_down, clear, cloudy, conventional_mine, cultivation, habitation, haze, partly_cloudy, primary, road, selective_logging, slash_burn, water


In [ ]:
# Display information about the test table
print('\nDisplaying information about the test table:')
print("Table header: \n", submission_df.head(), '\n')
print("Table info: ", submission_df.info(), '\n')
print("Null values: ", submission_df.isnull().sum(), '\n')

test_all_tags = []
for tags_str in submission_df["tags"].values:
    test_all_tags.extend(tags_str.split())
test_unique_tags = sorted(list(set(test_all_tags)))
print(f"\nNumber of tags in test data: {len(test_unique_tags)}")
print(f"Tags: {test_unique_tags}")

Displaying information about the test table. The table info shows that the test table does not have split tags, and each row contains 5 possible tags.

#### Tag Frequency Analysis
To get an overview of how many times each tag appears in the training table.

In [ ]:
tag_counts = {}
for tag in all_tags:
    if tag in tag_counts: tag_counts[tag] += 1
    else: tag_counts[tag] = 1

sorted_tags = sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)
print("\nMost frequent tags:")
for tag, count in sorted_tags[:10]:
    print(f"{tag}: {count} occurrences")

def plot_tag_distribution(sorted_tags):
    tags = [tag for tag, _ in sorted_tags]
    counts = [count for _, count in sorted_tags]
    plt.figure(figsize=(12, 5))
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(tags)))
    bars = plt.bar(range(len(tags)), counts, color=colors)
    plt.xlabel('Tags'); plt.ylabel('Number of occurrences'); plt.title('Distribution of individual tags in the training dataset')
    plt.xticks(range(len(tags)), tags, rotation=45, ha='right')
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.1, f'{int(height)}', ha='center', va='bottom')
    plt.tight_layout(); plt.savefig('tags_distribution_train.png'); plt.show(); plt.close()

plot_tag_distribution(sorted_tags)

### Training Data and Creating Classifiers

#### Creating One-Hot Encoding for Tags
In order to work with the image tags in the training model, I need to convert the data into vectors so I can work with them in PyTorch, where they are then converted via a tensor into a vector format of floats that the training model will work with. I print the dataset header and a new column is visible, where for each record a vector array is created with 17 float values, each representing either 1.0 (True) or 0.0 (False), depending on whether the image contains the given tags. One image can have multiple tags.


In [ ]:
# Creating one-hot encoding for tags
def get_tag_map(tags):
    labels = np.zeros(len(unique_tags))
    if pd.isna(tags): return labels
    for tag in tags.split():
        if tag in unique_tags:
            labels[unique_tags.index(tag)] = 1
    return labels
# Adding one-hot encoding to the dataframe
train_df['tag_vector'] = train_df['tags'].apply(get_tag_map)
print(train_df.head())

#### Creating a Dataset Class for PyTorch DataLoader
To be able to train data in ResNet50, I need to create a dataset class for PyTorch. In this class I define the dataframe I will work with, the directory for images, and the type of transformation for those images. There is a function that returns the length of the dataframe and a __getitem__ function that loads images and applies transformations. Using FloatTensor in PyTorch it converts the tag_vector column (containing tag vectors in float format) into a multi-dimensional matrix of the given data type for training the dataset, and then returns the created dataset for processing with images and the tag vector.


In [ ]:
# Creating a custom Dataset class for PyTorch
class PlanetDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['image_name']
        img_path = os.path.join(self.img_dir, img_name + '.jpg')
        # Loading image
        image = Image.open(img_path).convert('RGB')
        # Applying transformations if defined
        if self.transform: image = self.transform(image)
        # Getting one-hot encoding for tags
        tag_vector = torch.FloatTensor(self.dataframe.iloc[idx]['tag_vector'])
        return image, tag_vector

In [ ]:
# Calculating adequate mean and standard deviation for normalization
def calculate_mean_std(dataset):
    temp_transform = transforms.Compose([ transforms.Resize((224, 224)), transforms.ToTensor()])
    # Temporarily change dataset transformations
    original_transform = dataset.transform
    dataset.transform = temp_transform
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    mean = torch.zeros(3); std = torch.zeros(3) # create empty arrays of size 3 for std and mean
    total_samples = 0
    # calculating mean
    for data, _ in dataloader:
        batch_samples = data.size(0)
        data = data.view(batch_samples, data.size(1), -1)
        mean += data.mean(2).sum(0)
        total_samples += batch_samples
    mean = mean / total_samples
    # calculating std
    for data, _ in dataloader:
        batch_samples = data.size(0)
        data = data.view(batch_samples, data.size(1), -1)
        std += ((data - mean.unsqueeze(1))**2).sum([0,2])
    std = torch.sqrt(std / (total_samples * 224 * 224))
    # Restore original transformations
    dataset.transform = original_transform
    print(f"mean: {mean}, std: {std}")
    return mean.tolist(), std.tolist()
mean, std = calculate_mean_std(PlanetDataset(train_df, TRAIN_DIR, transform=None))

#### Defining Transformations
I define transformations for training and validation data that will be used for training the model. I can define separate transformations for training and validation data.

For transformations I can specify:
- The pixel size to start with.
- Whether to convert to tensor.
- Whether to randomly flip horizontally or vertically. This is very useful for images of this type, as the images can be taken from any angle.
- Random rotation, specifying the number of degrees.
- Image adjustments: brightness, contrast, saturation, and more...
- Adding GaussianBlur to simulate atmospheric conditions.
- Data normalization, where I specify the mean and standard deviation for three-dimensional data of transformed images. For example, when I created the algorithm to calculate mean and std for normalization, it calculated mean: 0.3117, 0.3408, 0.2991 and std: 0.1669, 0.1436, 0.1372
- And many more properties...


In [ ]:
# Defining image transformations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    #transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)), # gentle blur, sigma from 0.1 weak to 1.5 medium blur values
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

#### Splitting into Training Data and Preparing for Model Training
I will split the model into training and validation data. I set the split ratio to 80/20 for training and validation data, and set random_state for shuffling. I then print the number of training and validation samples. I create datasets for training and validation data using the PlanetDataset class defined for the train model via torch.


In [ ]:
train_data, valid_data = train_test_split(train_df, test_size=0.2, random_state=42)
print(f"\nNumber of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(valid_data)}")
# Creating datasets
train_dataset = PlanetDataset(train_data, TRAIN_DIR, transform=train_transform)
valid_dataset = PlanetDataset(valid_data, TRAIN_DIR, transform=val_transform)

def visualize_class_correlations(dataset, unique_tags):
    num_tags = len(unique_tags); correlation_matrix = np.zeros((num_tags, num_tags))
    # Collecting all labels
    all_labels = []
    for _, label in dataset:
        all_labels.append(label.numpy())
    all_labels = np.array(all_labels)
    # Calculating correlations between tags
    for i in range(num_tags):
        for j in range(num_tags):
            correlation_matrix[i, j] = np.corrcoef(all_labels[:, i], all_labels[:, j])[0, 1]
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, xticklabels=unique_tags, yticklabels=unique_tags, cmap='coolwarm', vmin=-1, vmax=1, annot=False)
    plt.title('Correlation between tags'); plt.tight_layout(); plt.show()
visualize_class_correlations(train_dataset, unique_tags)


Here I created a correlation matrix between classes to get an overview of some critical classes and their occurrence in the dataset.


In [ ]:
def set_critical_class_weights(unique_tags, tag_distribution, threshold=0.05):
    weights = torch.ones(len(unique_tags))
    ## Identifying very rare classes (less than 5% of data)
    #total_samples = sum(tag_distribution.values())
    #threshold_percentage = 0.05
    #critical_classes = []
    #for i, tag in enumerate(unique_tags):
    #    count = tag_distribution[tag]
    #    percentage = count / total_samples
    #    if percentage < threshold_percentage:
    #        weights[i] = 2.0  # Basic increase for rare classes
    #        critical_classes.append((i, tag, percentage))
    # Adding weights for domain-critical classes
    domain_critical = ["slash_burn","blow_down", "conventional_mine", "artisinal_mine", "blooming", "selective_logging","bare_ground","haze","habitation","cultivation"]
    for tag in domain_critical:
        if tag in unique_tags:
            idx = unique_tags.index(tag)
            weights[idx] = max(weights[idx], 2)  # Higher weight for domain-critical classes
    # Print the set weights
    print("Set higher weights for critical classes:")
    for i, tag in enumerate(unique_tags):
        if weights[i] > 1.0:
            print(f"{tag}: {weights[i]:.2f}")
    return weights

class_weights = set_critical_class_weights(unique_tags, tag_counts)

I set higher weights for some classes.

#### Loading Data into the DataLoader and Visualizing It.

I load the datasets into a DataLoader containing batches (iterations) of data using a PyTorch function, so I can build a training model from them. I specify the batch size (usually 32), whether to shuffle, and the number of worker threads for loading. Multi-threaded loading can sometimes cause issues such as longer loading times or deadlocks. I can then print a sample of one batch of image and label data from the train DataLoader. This operation can take a long time if there is a lot of data. I can print what one batch of data looks like, prepared for building the training model with ResNet50. I visualize some images from the dataset, after which the data is ready for training.


In [ ]:
# Creating data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
# Sample loading of one batch of data
start_time = time.time()
images, labels = next(iter(train_loader))
end_time = time.time()
print(f"\nShape of loaded images: {images.shape}")
print(f"Shape of loaded tags: {labels.shape}")
print("Time to load data into DataLoader: ", end_time - start_time, " s")
# Visualizing several images from the dataset
mean = mean; std = std #mean = [0.485, 0.456, 0.406]; std = [0.229, 0.224, 0.225]
def visualize_sample(dataset, num_samples=15):
    num_rows = (num_samples + 5 - 1) // 5
    plt.figure(figsize=(15, 3*num_rows))
    for i in range(num_samples):
        image, label = dataset[i]
        image = image.permute(1, 2, 0).numpy()
        # Denormalize image for display
        image = std * image + mean
        image = np.clip(image, 0, 1)
        tags = [unique_tags[j] for j in range(len(unique_tags)) if label[j] == 1]
        plt.subplot(num_rows, 5, i+1)
        plt.imshow(image)
        plt.title(f"Tags: {', '.join(tags)}", fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
visualize_sample(train_dataset)
print("\nData is ready for model training!")

#### Using a Pre-trained Model
We create a multi-label classifier class to classify the given data for training.

First, we perform convolutional preprocessing. For geographic data, this is beneficial due to positional invariance, since any image can be located anywhere. First, a 2D convolution can be applied (performed during preprocessing) that takes an input RGB image (3 channels) and outputs 64 feature maps. Kernel 3×3 with padding=1 → preserves dimensions; the purpose of the 2D convolution is to extract basic features (edges, textures). A ReLU function adds non-linearity enabling learning of complex patterns. We apply 2D max pooling on the input signal composed of several input planes, which halves the resolution, reduces computational cost, and preserves the most important features. This can be repeated multiple times to refine the extracted features and create more complex representations. I then removed the preprocessing since it degraded training performance.

The classifier will use the pre-trained ResNet50 model. Once the data is ready, I can start with the pre-trained ResNet50 model. During initialization, the super() method is called for the given class and __init__, which allows calling methods defined in the parent class from a subclass, enabling functionality inherited from the parent class to be extended and customized. We define a variable for the ResNet50 model with pre-trained data. This is what the ResNet50 architecture looks like:

<div align="center">
    <img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/0*9LqUp7XyEx1QNc6A.png" alt="resnet50 architecture" style="width: 70%; height: auto;">
</div>

Freezing all layers except the last ones — This is a technique used in transfer learning, especially for smaller datasets. Except for the last 10 parameters, it sets requires_grad = False. This means these parameters will not be updated during training. Lower CNN layers learn to recognize basic patterns (edges, textures, shapes). It prevents overfitting — with small datasets, training all parameters would lead to overfitting. Frozen layers act as "stable feature extractors". Only higher layers are trained, adapting to the specific task. This enables faster training (fewer parameters to update = faster backward pass, lower memory requirements) and gradual unfreezing (the last 10 parameters remain trainable, allowing fine-tuning of higher layers for the specific task).

*Strategy by dataset size:*

*Small dataset:* Freeze most layers (as in the code)

*Medium dataset:* Freeze only lower layers

*Large dataset:* Train the whole model with a lower learning rate

We remove the original avgpool and fc layers and replace them with a backbone, then add AdaptiveAvgPool2d() to it. We can replace the model's original avgpooling with global average pooling using the AdaptiveAvgPool2d(output_size=(1, 1)) class — an adaptive averaging layer that always produces an output of size 1×1 regardless of input size. The output_size parameter can be set to a higher value, e.g. (7, 7), giving an output of size 7×7. Its advantages: size invariance — satellite images can have different resolutions, and GAP handles this; less overfitting — fewer parameters than large FC layers (dense layers); spatial robustness — objects can be anywhere in the satellite image; interpretability — each channel represents one "concept" (the degree to which a person can understand the reasons behind model predictions or decisions).

We have Input: [batch, channels, H, W] - any H, and Output: [batch, channels, 1, 1] - always 1×1 when set to 1×1.

How it works: The layer calculates the kernel size itself. Either kernel_size = (H, W) - covers the entire feature map or stride = (H, W) - no overlap. Averaging: For each feature map it calculates the average of all values. The result is one number representing the entire feature map.

<div align="center">
    <img src="gap_diagram.svg" alt="Global Average Pooling Diagram" style="max-width: 100%; height: auto;">
</div>

We define the sequence in which we want the model to train data. If global average pooling is set, the layer still needs to be flattened.
<div align="center">
    <img src="planet_classifier_diagram.svg" alt="Global Average Pooling Diagram" style="width: 70%; height: auto;">
</div>

Channels represent different dimensions or "views" of the input data. In image data, each channel contains a specific type of information about the image. RGB channels are most commonly used. Convolutional filters process channels. Result: a combination of information from all channels, and after each convolutional layer new "channels" (feature maps) are created.

What channels are used for:
1. Detecting different features - Low-level: Edges, textures, colors; High-level: Shapes, objects, patterns
2. Channel specialization - During training, channels "learn" to recognize: Channel 1: Horizontal edges, Channel 2: Vertical edges, Channel 3: Specific textures, Channel 4: Color patterns

We first define a linear transformation for the input data according to the equation for a fully connected layer, where we specify the number of dimensions (2048 is used for GAP) and the size of each input and output sample. Optionally, a bias boolean can be set to determine whether the layer learns an additive bias (default is True).

We apply batch normalization for 2D or 3D input using BatchNorm1d to accelerate training of deep networks by reducing internal covariate shift. This is especially useful for geographic images.

Next, ReLU applies the rectified linear unit function element-wise according to the given formula. We have an inplace bool that determines whether to perform the operation in place (default is False).

We define the Dropout, which zeros some elements of the input tensor with probability p during training. The zeroed elements are chosen independently for each forward call and sampled from a Bernoulli distribution. Each channel is zeroed independently during the forward call. This is an effective technique for regularization and preventing co-adaptation of neurons. The module simply computes the unit function. It has parameters p (float) indicating the probability of zeroing an element (default 0.5) and inplace bool for in-place operation (default False).

We can then repeat ReLU, Dropout, and a linear transformation that converts the number of input dimensions to the number of classes. At the end of the sequence, a final linear transformation is applied to convert the input dimensions to the output number of classes.

The class also has a function to return the forward layer of the classifier for input values x.


In [ ]:
# Creating a multi-label classifier using ResNet50
class PlanetClassifier(nn.Module):
    def __init__(self, num_classes):
        super(PlanetClassifier, self).__init__()
        #self.preprocessing = nn.Sequential(
        #    # Normalization of spectral bands
        #    nn.Conv2d(3, 64, kernel_size=1),  # 1x1 conv for spectral mix
        #    nn.BatchNorm2d(64), nn.ReLU(),
        #    # Spatial filters specific to satellite data
        #    nn.Conv2d(64, 64, kernel_size=5, padding=2),  # Larger kernel for spatial patterns
        #    nn.BatchNorm2d(64), nn.ReLU(), nn.Conv2d(64, 3, kernel_size=1),  # Back to 3 channels
        #)
        self.resnet = models.resnet50(pretrained=True)
        #self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        for param in list(self.resnet.parameters())[:-10]: param.requires_grad = False # only for smaller datasets
        # Remove original avgpool and fc layers
        self.backbone = nn.Sequential(*list(self.resnet.children())[:-2])
        # Global Average Pooling instead of standard
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=(1, 1)) # [batch, 2048, 2, 2] last layer of ResNet50, reduces output to 2x2
        #in_features = self.resnet.fc.in_features # Replacing the final fully connected layer for multi-label classification
        spatial_features = 2048 # 2048 channels, each with 2x2 spatial features
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(spatial_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        features = self.backbone(x)  # [batch, 2048, H, W], Input image x passes through the backbone network, typically a pre-trained CNN (e.g. ResNet, VGG)
        pooled = self.global_avg_pool(features)  # [batch, 2048, 1, 1]
        output = self.classifier(pooled)  # [batch, num_classes], applying the sequence
        return output


In [ ]:
# Multi-label classifier using EfficientNet_b0
class EfficientNetGeoClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_layers=True):
        super(EfficientNetGeoClassifier, self).__init__()
        self.efficientnet = models.efficientnet_b0(pretrained=pretrained)
        # Freezing the first layers
        #if freeze_layers:
        #   for param in self.efficientnet.features.parameters():
        #       param.requires_grad = False
        # Removing the original classifier
        self.features = self.efficientnet.features
        self.avgpool = self.efficientnet.avgpool
        # Get the number of output features
        num_features = self.efficientnet.classifier[1].in_features
        #print(f"Number of output features: {num_features}")
        # Replace classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),  # Dropout at the beginning
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),  # Higher dropout
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        # Feature extraction
        x = self.features(x)
        x = self.avgpool(x)
        # Classification
        x = self.classifier(x)
        return x

#### Creating the Training Model
We create a function for training the model. It takes our created classifier, the training data dataset loader, the validation data dataset loader, criterion for recording loss, optimizer (Adam is most common), scheduler (records the ongoing learning rate), and epochs (how many epochs to run).

First, I define the device for training the model — whether to use CUDA (Nvidia GPU) or, if the computer has no Nvidia GPU, CPU.

Next, I assign this device to our model for learning. I define a variable to track the best value, based on which I save the best model at the end.

I create lists for training and validation losses, based on which the model's performance will be displayed.

The model will be trained in epochs (cycles) using a main for-loop for the desired number of epochs. Before the next for-loop, we print which epoch the model is on.

**1. Training Data**
Using model.train(), training begins in the given model phase. During training, the loss is recorded in the running_loss variable. A for-loop iterates through inputs and labels for the training data loader, adding them to the device memory (GPU or CPU) for learning. The optimizer's zero_grad() resets the gradients of all optimized tensor objects. The inputs are passed through the model to get outputs, and then a loss variable is created using the criterion with outputs and labels. loss.backward() is called to compute the gradient of the current tensor with respect to graph leaves. Then optimizer.step() performs one optimization step to update the parameters. Finally, running_loss is updated by the loss item multiplied by the input size.

When this for-loop completes for a given epoch, we create the epoch_train_loss variable storing the percentage loss for that epoch, then append it to the train_losses list.

**2. Validation Data**
Next, validation data must be processed in the given phase, where model evaluation begins using model.eval(). Again, running_loss is tracked, and lists are maintained for storing predictions and labels. Using PyTorch with gradient computation disabled, a for-loop iterates through inputs and labels for the validation data loader, performing the validation phase, which is simpler than training. Again, inputs and labels are added to the device, outputs are computed, and the loss variable is created using criterion, with running_loss updated for the current validation loss. For validation data, binary predictions are created using sigmoid on the outputs with a minimum threshold for predictions. The binary predictions are also converted to float format and stored in the all_predictions list (returned from CPU memory). Labels are stored in the all_labels list (also returned from CPU memory).

The predictions and labels from validation data are converted using torch to 1D dimensions and NumPy array format. The epoch_val_loss variable stores the validation accuracy.

**Evaluating the Given Phase**
After training and validation, we evaluate the current phase of the training model.

From the given data, we compute precision score and recall score. The F1 score for the phase and overall are calculated, then printed. F1 scores for the 5 best and worst tags are also printed.

The learning rate is updated using the scheduler (important). The best model is saved to a .pth file.

**Visualizing Training Model Results**
Finally, we create a visualization chart of the training model results, save it to a file, and return the best model from the function.


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=10, model_name='ResNet50'):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = model.to(device)
    best_val_f1 = 0.0
    train_losses = []; val_losses = []; train_F1_scores = []; val_F1_scores = []
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 40)
        model.train()
        running_loss = 0.0; all_train_preds = []; all_train_labels = []
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            # for multilabel classification
            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_train_preds.append(preds.cpu())
            all_train_labels.append(labels.cpu())
        epoch_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        all_train_preds, all_train_labels = torch.cat(all_train_preds, dim=0).numpy(), torch.cat(all_train_labels, dim=0).numpy()
        train_f1 = f1_score(all_train_labels, all_train_preds, average='samples', zero_division=0)
        train_F1_scores.append(train_f1)

        model.eval()
        running_loss = 0.0; all_preds = []; all_labels = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())
        epoch_val_loss = running_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)

        all_preds, all_labels = torch.cat(all_preds, dim=0).numpy(), torch.cat(all_labels, dim=0).numpy()
        # Calculating metrics
        precision = precision_score(all_labels, all_preds, average='samples', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='samples', zero_division=0)
        sample_f1 = f1_score(all_labels, all_preds, average='samples', zero_division=0)
        macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        val_F1_scores.append(sample_f1)
        print(f'Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Train F1: {train_f1:.4f}, Val F1: {sample_f1:.4f}')
        print(f'Precision: {precision:.4f}, Real Precision(Recall): {recall:.4f}')
        print(f'Sample F1(Harmonic mean): {sample_f1:.4f}, Macro F1(Score for every class): {macro_f1:.4f}')
        tag_f1_scores = []
        for i, tag in enumerate(unique_tags):
            tag_f1 = f1_score(all_labels[:, i], all_preds[:, i], zero_division=0)
            tag_f1_scores.append((tag, tag_f1))
        tag_f1_scores.sort(key=lambda x: x[1], reverse=True)
        print("\nF1 scores for the best tags:")
        for tag, f1 in tag_f1_scores[:5]:
            print(f"{tag}: {f1:.4f}")
        print("\nF1 scores for the worst tags:")
        for tag, f1 in tag_f1_scores[-10:]:
            print(f"{tag}: {f1:.4f}")

        # Updating learning rate
        #scheduler.step()
        scheduler.step(epoch_val_loss)

        if sample_f1 > best_val_f1:
            best_val_f1 = sample_f1
            torch.save(model.state_dict(), f"best_planet_classifier_{model_name}.pth")
            print("Best model saved!")
    # Visualizing training results
    plt.figure(figsize=(15, 5))
    # Loss chart
    plt.subplot(1, 2, 1)
    plt.plot(range(1, num_epochs+1), train_losses, 'b-', label='Training loss')
    plt.plot(range(1, num_epochs+1), val_losses, 'r-', label='Validation loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss (%)'); plt.title('Training and validation loss')
    plt.grid(True)
    plt.legend()
    # Accuracy chart
    plt.subplot(1, 2, 2)
    plt.plot(range(1, num_epochs+1), train_F1_scores, 'g-', label='Training F1 score')
    plt.plot(range(1, num_epochs+1), val_F1_scores, 'm-', label='Validation F1 score')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.title('Training and validation F1 score')
    plt.grid(True)
    plt.legend()
    plt.tight_layout(); plt.savefig('training_results.png', dpi=300, bbox_inches='tight'); plt.show()
    return model

### Training the ResNet50 Model
Now we perform model training. We create the model from our custom classifier class, passing the number of classes equal to the number of unique tags in the training dataset, which also indicates multi-label classification.

We define the criterion for the loss. BCEWithLogitsLoss() combines the sigmoid layer and BCELoss into one class and is ideal for multi-label classification where one image can contain multiple elements at once. This version of the loss is numerically more stable than using plain sigmoid, as it combines operations into a single layer using the log-sum-exp trick for numerical stability. For this loss, we can specify reduction (the reduction to apply to the output: mean, sum, or none), pos_weight (an optional tensor with elements torch.ones([num_classes]) for different classes in a multi-label binary classification scenario; each element in pos_weight is designed to adjust the loss function based on the imbalance between negative and positive samples for the respective class; useful for datasets with varying class imbalance), weight (an optional tensor for manual weight scaling of each element's loss in the batch), size_average (optional bool, deprecated; by default, losses are averaged or summed per batch depending on size_average), and pos_weight (optional tensor specifying the weight of positive examples).

We define the optimizer, which implements the optimization algorithm — we create an object that maintains the current state and updates parameters based on computed gradients. Adam is most commonly used, which adjusts the learning rate for each parameter individually and has many configurable properties: params (iterable of parameters), lr (float and tensor; learning rate), betas (coefficients for computing running averages of gradient and its square), eps (float added to the denominator for numerical stability), weight_decay (float weight decay), decoupled_weight_decay (bool whether to accumulate weight decay in momentum or variance), amsgrad (whether to use the AMSGrad variant), foreach (bool), maximize (whether to maximize the objective with respect to parameters instead of minimizing), etc. It takes a filter that only selects model parameters with enabled gradients. This setup allows fast learning at the beginning of training and finer tuning as the model approaches the optimum, which is particularly important for detailed classification of geographic features.

The ReduceLROnPlateau() scheduler reduces the learning rate when the validation loss stops improving. For geographic data where it can be difficult to find the optimal learning rate, this adaptive adjustment is very useful. It has the following parameters: min (tracks minimum metrics, validation loss), patience (waits a given number of epochs without improvement before reducing the learning rate), factor (when activated, reduces the learning rate by half (0.5 × current value)).

All these variables are then passed to the train_model function which trains the model. The function takes: the model classifier, the training data loader, the validation data loader, the criterion for loss, the optimizer, the scheduler, and the set number of epochs (phases) to run.


In [ ]:
# Creating a Focal Loss criterion class that better tracks weights of critical classes
import torch.nn.functional as F
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, pos_weight=self.pos_weight, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss
        return focal_loss.mean()

In [ ]:
# Initializing model, criterion, optimizer, and scheduler
num_epochs = 15
model = PlanetClassifier(num_classes=len(unique_tags)) # ResNet50 model
#criterion = nn.BCEWithLogitsLoss() #pos_weight=class_weights
criterion = FocalLoss(alpha=1, gamma=2, pos_weight=class_weights) # Focal loss for class balancing and focusing on hard cases
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4, betas=(0.9, 0.999)) # AdamW is an improved version of Adam with weight regularization
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs, T_mult=1, eta_min=1e-6, last_epoch=-1)

print("Training the model...")
start_time = time.time()
trained_model = train_model(model=model, train_loader=train_loader, val_loader=valid_loader, criterion=criterion, optimizer=optimizer, scheduler=scheduler, num_epochs=num_epochs)
end_time = time.time()
print(f"Time to train the model in {num_epochs} epochs: ", (end_time - start_time)/60, " minutes, ", (end_time - start_time)/60/60, " hours")


The first training run took nearly 29 hours in total; subtracting about half of the time the laptop was in hibernation, it was around 15 hours. During the first training run I achieved approximately 89% on training data and approximately 89.5% on validation data. See the curve showing the progress of training in individual phases, where the validation data was variable. During the first training run I encountered the problem that some classes always had an F1 score of 0, making it impossible for the machine learning model to detect them, although during the run some of those zero tags did acquire an F1 score. It seems that it would not be a bad idea to increase the probabilities for some critical values in the dataset that appear infrequently, so the machine learning model can better learn to detect them, as my Macro F1 score is weak.

I then adjusted it a bit, trying various improvements such as convolution or increasing weights for critical classes. In the end, I added global average pooling with just one layer and the loss dropped to 10% while the Macro score increased from 50 to 56%.

I also tried changing the optimizer to AdamW, which is better than Adam because it allows weight regularization (additionally, weight_decay and betas — coefficients used for computing running averages of gradient and its square — can be set here). I changed the scheduler to OneCycleLR (which also takes max_lr, steps_per_epoch, epochs, pct_start — the percentage of the warm-up phase spent increasing lr, div_factor — determines the initial learning rate initial_lr = max_lr/div_factor, default 25; final_div_factor — determines the minimum learning rate via min_lr = initial_lr/final_div_factor, default 1e4; and many more configurable parameters), which should have a high learning rate in the middle and thus quickly discover rare classes. I also changed the criterion to FocalLoss (parameters: alpha — balancing between positive and negative examples; gamma — balancing between easy and hard examples; pos_weight), which should better recognize imbalanced classes and automatically focuses more on "hard" samples → classes with F1=0 receive more attention. The classes with increased weights also feed into it so the model can better learn them. The loss dropped to 4% and the Macro score increased to 61%.

#### EfficientNet-b0 Model

In [ ]:
model_2 = EfficientNetGeoClassifier(num_classes=len(unique_tags)) # EfficientNet model
criterion = FocalLoss(alpha=1, gamma=2, pos_weight=None)# pos_weight=class_weights  # Suitable for multi-label classification, with increased weights for critical classes
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-5, betas=(0.9,0.999))
num_epochs = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs,eta_min=1e-6)

print("Training the model...")
start_time = time.time()
trained_model_eficientnet = train_model(model=model_2, train_loader=train_loader, val_loader=valid_loader, criterion=criterion, optimizer=optimizer, scheduler=scheduler, num_epochs=num_epochs, model_name="EfficientNet_b0")
end_time = time.time()
print(f"Time to train the model in {num_epochs} epochs: ", (end_time - start_time)/60, " minutes, ", (end_time - start_time)/60/60, " hours")


I wanted to try a different model, EfficientNet-b0, but it did not converge at all — even in the first and third epochs the loss was at 70%.

#### ResNet50 without Increased Weights
The loss criterion remains the same, just without the input weights. AdamW also stays the same. I changed the scheduler to CosineAnnealingWarmRestarts.


In [ ]:
# Initializing model, criterion, optimizer, and scheduler
num_epochs = 15
model = PlanetClassifier(num_classes=len(unique_tags)) # ResNet50 model
#criterion = nn.BCEWithLogitsLoss() #pos_weight=class_weights
criterion = FocalLoss(alpha=1, gamma=2, pos_weight=None) # Focal loss for class balancing and focusing on hard cases
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4, betas=(0.9, 0.999)) # AdamW is an improved version of Adam with weight regularization
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs, T_mult=1, eta_min=1e-6, last_epoch=-1)

print("Training the model...")
start_time = time.time()
trained_model = train_model(model=model, train_loader=train_loader, val_loader=valid_loader, criterion=criterion, optimizer=optimizer, scheduler=scheduler, num_epochs=num_epochs, model_name="ResNet50_non_weighted")
end_time = time.time()
print(f"Time to train the model in {num_epochs} epochs: ", (end_time - start_time)/60, " minutes, ", (end_time - start_time)/60/60, " hours")

Training without increased weights gave slightly better results based on Sample F1 Score and precision.

### Evaluating the Best Model
We load the best training model using the given model class, along with the required data: unique_tags, valid_data, valid_loader, which we use to perform analysis and plot confusion matrices.

In [ ]:
all_tags = []
for tags in train_df['tags'].values:
    all_tags.extend(tags.split())
unique_tags = sorted(list(set(all_tags)))

def get_tag_map(tags):
    labels = np.zeros(len(unique_tags))
    if pd.isna(tags): return labels
    for tag in tags.split():
        if tag in unique_tags:
            labels[unique_tags.index(tag)] = 1
    return labels
# Adding one-hot encoding to the dataframe
train_df['tag_vector'] = train_df['tags'].apply(get_tag_map)

model = PlanetClassifier(num_classes=len(unique_tags))
model.load_state_dict(torch.load('best_planet_classifier_ResNet50_non_weighted.pth'))
# loading the validation dataset
_ , valid_data = train_test_split(train_df, test_size=0.2, random_state=42)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3117, 0.3408, 0.2991], std=[0.1669, 0.1436, 0.1372])
])

valid_dataset = PlanetDataset(valid_data, TRAIN_DIR, transform=val_transform)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=0)


We can also try loading the model with increased weights.


In [ ]:
model.load_state_dict(torch.load('best_planet_classifier_ResNet50.pth'))

#### Testing the Model
A function for prediction and creating a submission file for test data using the best model. Here I create the submission file, where my model makes predictions for the images.

In [ ]:
def create_submission(model, test_loader, max_samples):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    predictions = []; image_names = []; processed_samples = 0
    with torch.no_grad():
        for batch, (inputs, _) in enumerate(test_loader):
            if processed_samples >=max_samples: break
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float().cpu().numpy()
            for i in range(len(preds)):
                idx = batch * test_loader.batch_size + i
                if processed_samples < max_samples:
                    img_name = submission_df.iloc[idx]['image_name']
                    image_names.append(img_name)
                    # Getting tags for prediction
                    pred_tags = []
                    for j, val in enumerate(preds[i]):
                        if val == 1:
                            pred_tags.append(unique_tags[j])
                    predictions.append(' '.join(pred_tags))
                    processed_samples += 1
    print(f"Processed {processed_samples} samples.")
    # Creating submission dataframe
    submit_df = pd.DataFrame({'image_name': image_names, 'tags': predictions })
    # Saving to CSV
    submit_df.to_csv('submission.csv', index=False)
    print("Submission file has been created!")

# loading the test dataset
filtered_submission_df = submission_df.head(10000)
filtered_submission_df['tag_vector'] = [np.zeros(len(unique_tags)) for _ in range(len(filtered_submission_df))]
test_dataset = PlanetDataset(filtered_submission_df, TEST_DIR, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

start_time = time.time()
create_submission(trained_model, test_loader, max_samples=10000)
end_time = time.time()
print(f"Time to create submission on test data: ", (end_time - start_time)/60, " minutes, ", (end_time - start_time)/60/60, " hours")


#### Visualizing Predictions from the Submission File
Even though I don't have original tags for the test data and therefore can't perform any evaluation of that data, I would at least like to visualize some images and their predictions to see if the model output actually corresponds to what is in the image.


In [ ]:
def visualize_predictions(model, test_loader, submission_df, test_dir, unique_tags, num_samples=12, figsize=(15, 12), threshold=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    # Prepare a list for storing results
    results = []; processed_samples = 0
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            if processed_samples >= num_samples: break
            # Unpack batch based on its contents
            if len(batch) == 2: inputs, _ = batch  # ignore dummy labels
            else: inputs = batch[0]
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float().cpu().numpy()
            # Processing each image in the batch
            for i in range(len(preds)):
                if processed_samples >= num_samples: break
                # Getting the image index in the original dataset
                global_idx = batch_idx * test_loader.batch_size + i
                if global_idx < len(submission_df):
                    img_name = submission_df.iloc[global_idx]['image_name']
                    # Getting predicted tags
                    pred_tags = []; pred_probs = []
                    for j, val in enumerate(preds[i]):
                        if val == 1:
                            pred_tags.append(unique_tags[j])
                            pred_probs.append(probs[i][j].item())
                    # If no tags, take top 3 probabilities
                    if not pred_tags:
                        top_indices = np.argsort(probs[i].cpu().numpy())[-3:][::-1]
                        for idx in top_indices:
                            pred_tags.append(f"{unique_tags[idx]} ({probs[i][idx].item():.3f})")
                    results.append({'image_name': img_name, 'predicted_tags': pred_tags, 'probabilities': pred_probs })
                    processed_samples += 1
    # Creating visualization
    rows = (num_samples + 3) // 6
    cols = min(6, num_samples)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1:
        axes = axes.reshape(1, -1)
    for idx, result in enumerate(results):
        row = idx // cols
        col = idx % cols
        img_path = os.path.join(test_dir, result['image_name'] + '.jpg')
        image = Image.open(img_path).convert('RGB')
        axes[row, col].imshow(image)
        axes[row, col].axis('off')
        if result['predicted_tags']:
            if result['probabilities']:
                title_parts = []
                for tag, prob in zip(result['predicted_tags'], result['probabilities']):
                    title_parts.append(f"{tag} ({prob:.2f})")
                title = ', '.join(title_parts[:2])  # max 2 tags
                if len(title_parts)>2:
                    title += ', '
                    title+="\n".join(title_parts[3:5])
            else:
                title = '\n'.join(result['predicted_tags'][:3]) # Just tag names
        else:
            title = "No tags"
        axes[row, col].set_title(f"{result['image_name']}\n{title}", fontsize=10, pad=10)

    # Hiding empty subplots
    for idx in range(len(results), rows * cols):
        row = idx // cols
        col = idx % cols
        axes[row, col].axis('off')
    plt.tight_layout(); plt.suptitle('Predicted tags on submission data', fontsize=16, y=1.02); plt.show()

visualize_predictions(trained_model, test_loader, filtered_submission_df, TEST_DIR, unique_tags, num_samples=18, figsize=(20, 12), threshold=0.5)

I think the model recognizes the images well. It can label everything where there is: Primary — forest, agriculture — some field, habitation — some houses, cloudy — overcast, partly_cloudy — partly cloudy, clear — clear sky, or road — a road.

#### Confusion Matrices on Validation Data
Now I will display confusion matrices using the validation data. A confusion matrix for each tag and then an overall confusion matrix.


In [ ]:
def evaluate_model_with_confusion_matrix(model, data_loader, unique_tags):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    all_preds = []; all_labels = []; processed_samples = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    # Merging all batches
    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()

    # GRID LAYOUT - All tags in one image
    n_tags = len(unique_tags)
    n_cols = 5  # Number of columns in the grid
    n_rows = (n_tags + n_cols - 1) // n_cols  # Calculating the number of rows
    plt.figure(figsize=(16, 3 * n_rows))
    binary_cm = multilabel_confusion_matrix(all_labels, all_preds) # true values, predictions
    for i in range(n_tags):
        plt.subplot(n_rows, n_cols, i + 1)
        cm = binary_cm[i]
        sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'], cbar=False)
        plt.title(f'{unique_tags[i]}', fontsize=10)
        plt.xlabel('')
        plt.ylabel('')
    plt.subplots_adjust(wspace=0.3, hspace=0.3)
    plt.tight_layout(); plt.savefig('all_tags_confusion_matrix_grid.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close()
    # COMPACT CONFUSION MATRIX - Aggregated values
    # Calculating aggregated values
    total_tp = np.sum([binary_cm[i][1,1] for i in range(n_tags)])
    total_fp = np.sum([binary_cm[i][0,1] for i in range(n_tags)])
    total_fn = np.sum([binary_cm[i][1,0] for i in range(n_tags)])
    total_tn = np.sum([binary_cm[i][0,0] for i in range(n_tags)])
    # Creating aggregated confusion matrix
    aggregated_cm = np.array([[total_tn, total_fp], [total_fn, total_tp]])
    plt.figure(figsize=(8, 6))
    sns.heatmap(aggregated_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Negative', 'Predicted Positive'], yticklabels=['Actual Negative', 'Actual Positive'])
    plt.title('Aggregated Confusion Matrix (all tags)', fontsize=14); plt.ylabel('Actual values'); plt.xlabel('Predicted values')
    # Adding percentages
    total = np.sum(aggregated_cm)
    for i in range(2):
        for j in range(2):
            percentage = (aggregated_cm[i,j] / total) * 100
            plt.text(j+0.5, i+0.7, f'({percentage:.1f}%)', ha='center', va='center', fontsize=10, color='red')
    plt.tight_layout(); plt.savefig('aggregated_confusion_matrix.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close()
    # 5. DETAILED TABLE
    print("="*80)
    print("DETAILED OVERVIEW OF RESULTS FOR ALL TAGS")

    precisions = []; recalls = []; f1_scores = []; supports = []
    for i in range(n_tags):
        cm = binary_cm[i]
        tn, fp, fn, tp = cm.ravel()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        support = tp + fn
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        supports.append(support)
    metrics_df = pd.DataFrame({'Tag': unique_tags, 'Precision': precisions, 'Recall': recalls, 'F1-Score': f1_scores, 'Support': supports})
    metrics_df_sorted = metrics_df.sort_values('F1-Score', ascending=False)
    print(f"{'Rank':<4} {'Tag':<20} {'Precision':<10} {'Recall':<8} {'F1-Score':<9} {'Support':<8}")
    print("-" * 70)
    for idx, (_, row) in enumerate(metrics_df_sorted.iterrows()):
        print(f"{idx+1:<4} {row['Tag']:<20} {row['Precision']:<10.3f} {row['Recall']:<8.3f} {row['F1-Score']:<9.3f} {row['Support']:<8.0f}")
    # Overall statistics
    avg_precision = np.mean(precisions)
    avg_recall = np.mean(recalls)
    avg_f1 = np.mean(f1_scores)
    print("\n"+"="*50)
    print("OVERALL STATISTICS")
    print(f"Average Precision: {avg_precision:.4f}")
    print(f"Average Recall:    {avg_recall:.4f}")
    print(f"Average F1 Score:  {avg_f1:.4f}")
    print(f"Total number of tags: {len(unique_tags)}")
    return metrics_df_sorted

start_time = time.time()
df_metrics = evaluate_model_with_confusion_matrix(model, valid_loader, unique_tags)
end_time = time.time()
print(f"Time to create confusion matrices on test data: ", (end_time - start_time)/60, " minutes, ", (end_time - start_time)/60/60, " hours")


#### Testing the Model on Created Test Data from Validation Data
Now I will create data with real labels from the validation data, where the correct name and the model's predicted value will be included. I can then compare them.

In [ ]:
# Class for loading the dataset for test data
class TestDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe
        self.img_dir = img_dir
        self.transform = transform
        # Check for existence of the tags column during initialization
        if 'tags' not in self.data.columns:
            raise ValueError("Missing column 'tags' in the CSV file for test data.")
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['image_name']
        img_path = os.path.join(self.img_dir, img_name+".jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        # Loading tags as a string (space-separated)
        tags_string = self.data.iloc[idx]['tags']
        # tags_list = tags_string.split() if isinstance(tags_string, str) else []
        return image, tags_string, img_name

# Testing the model on submission data
def test_model_multilabel(model, test_loader, threshold=0.5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    all_predictions = []; all_labels = []; all_image_names = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            if len(batch) == 3:
                images, labels, img_names = batch
                # For multi-label, store original tags
                all_labels.extend(labels)
            else:
                print("No labels")
                return
            images = images.to(device)
            outputs = model(images)
            # For multi-label classification use sigmoid
            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities > threshold).float()

            all_predictions.extend(predictions.cpu().numpy())
            all_image_names.extend(img_names)

    return all_predictions, all_labels, all_image_names

# Splitting validation data into test data
_, test_data = train_test_split(valid_data, test_size=0.5, random_state=42)
# Testing the model on test data
test_dataset = TestDataset(test_data, TRAIN_DIR, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
predictions, true_labels, image_names = test_model_multilabel(model, test_loader)

#### Model Evaluation on Test Data and Displaying Misclassified Images


In [ ]:
def analyze_results(y_true, y_pred, class_names, image_names=None):
    # Converting labels to the correct format
    mlb = MultiLabelBinarizer(classes=class_names)
    mlb.fit([class_names])
    if isinstance(y_true[0], str):
        # If they are strings with space-separated tags
        y_true_lists = []
        for label in y_true:
            tags = label.split() if label.strip() else []
            # Filter only known tags
            valid_tags = [tag for tag in tags if tag in unique_tags]
            y_true_lists.append(valid_tags)
        y_true = mlb.transform(y_true_lists)
    elif isinstance(y_true, np.ndarray) and y_true.ndim == 2:
        # Already a binary matrix
        y_true = y_true
    else:
        print("Converted to a different format")
        y_true = mlb.transform(y_true)

    accuracy = accuracy_score(y_true, y_pred)
    print(f"Overall accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nDetailed metrics for each class:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    # Find misclassified images
    if image_names is not None:
        misclassified = []
        for i, (true_label, pred_label, img_name) in enumerate(zip(y_true, y_pred, image_names)):
            pred_label = np.array(pred_label, dtype=int)
            if not np.array_equal(true_label, pred_label):
                #print(f"Misclassified image: {img_name}")
                misclassified.append({
                    'image': img_name,
                    'true_class': ", ".join([class_names[idx] for idx, val in enumerate(true_label) if val == 1]),
                    'predicted_class': ", ".join([class_names[idx] for idx, val in enumerate(pred_label) if val == 1 and true_label[idx] == 0])
                })
        print(f"\nNumber of misclassified images: {len(misclassified)}")
        # Show first 10 misclassified
        if misclassified:
            print("\nFirst 15 misclassified images:")
            for i, item in enumerate(misclassified[:15]):
                print(f"{i+1}. {item['image']}: {item['true_class']} → {item['predicted_class']}")
        return misclassified
    return None

def visualize_misclassified_samples(misclassified_list, img_dir, n_samples=15):
    if not misclassified_list:
        print("No misclassified images!")
        return
    n_samples = min(n_samples, len(misclassified_list))
    fig, axes = plt.subplots(3, n_samples//3, figsize=(18, 10))
    axes = axes.flatten()
    for i in range(n_samples):
        img_info = misclassified_list[i]
        img_path = os.path.join(img_dir, img_info['image']+".jpg")
        try:
            img = Image.open(img_path)
            axes[i].imshow(img)
            axes[i].set_title(f"True: {img_info['true_class']}\nPred: {img_info['predicted_class']}",fontsize=10)
            axes[i].axis('off')
        except Exception as e:
            axes[i].text(0.5, 0.5, f"Error loading\n{img_info['image']}",ha='center', va='center')
            axes[i].axis('off')

    plt.tight_layout(); plt.suptitle('Incorrectly predicted images', fontsize=16, y=1.02); plt.show()

if true_labels:
    # analyze results
    misclassified = analyze_results(true_labels, predictions, unique_tags, image_names=image_names)
    # visualize misclassified images
    visualize_misclassified_samples(misclassified,TRAIN_DIR)


As I found, some images may simply be incorrectly labeled or have nothing in them.